# Week 3: Classical Machine Learning Model Training

This notebook trains and evaluates classical machine learning models for the TruthLens fake news detector. The workflow includes data loading, preprocessing, TF-IDF feature extraction, model training, evaluation, and model artifact saving.

## 1. Setup and imports

In [ ]:
import os
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

# Ensure the project root is on Python path when running inside Jupyter notebooks
PROJECT_ROOT = None
for candidate in [Path.cwd()] + list(Path.cwd().parents):
    if (candidate / 'backend').exists() and (candidate / 'data').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate project root containing backend/ and data/')
sys.path.insert(0, str(PROJECT_ROOT))

from backend.preprocess import build_tfidf, clean_text, download_nltk_resources, load_dataset

sns.set(style='whitegrid', palette='muted', font_scale=1.05)
DATA_DIR = PROJECT_ROOT / 'data'
MODEL_DIR = PROJECT_ROOT / 'backend' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

ModuleNotFoundError: No module named 'backend'

### Download NLTK resources

In [ ]:
download_nltk_resources()

## 2. Load and preprocess data

In [ ]:
df = load_dataset(DATA_DIR)
df = df[['title', 'text', 'label']].copy()
df['title'] = df['title'].fillna('')
df['text'] = df['text'].fillna('')
df['content'] = df['title'] + ' ' + df['text']

print('Dataset shape:', df.shape)
print('Label distribution:')
print(df['label'].value_counts(normalize=True).rename('ratio'))
df.head(3)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='label', data=df)
plt.title('Fake News (1) vs Real News (0)')
plt.xlabel('Label')
plt.ylabel('Count')
plt.show()

### Clean the combined text

In [ ]:
cleaned_texts = []
for text in df['content'].tolist():
    cleaned_texts.append(clean_text(text))

df['cleaned_text'] = cleaned_texts

df[['content', 'cleaned_text']].head(3)

In [ ]:
df.to_csv(DATA_DIR / 'cleaned_news.csv', index=False)
print('Saved cleaned dataset to', DATA_DIR / 'cleaned_news.csv')

## 3. TF-IDF feature engineering

In [ ]:
vectorizer = build_tfidf(df['cleaned_text'].tolist(), max_features=5000, ngram_range=(1, 2))
X = vectorizer.transform(df['cleaned_text'].tolist())
y = df['label'].values

print('TF-IDF features shape:', X.shape)
joblib.dump(vectorizer, MODEL_DIR / 'tfidf_vectorizer.joblib')
print('Saved TF-IDF vectorizer to', MODEL_DIR / 'tfidf_vectorizer.joblib')

## 4. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print('Training shape:', X_train.shape)
print('Test shape:', X_test.shape)

## 5. Train classical models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, solver='liblinear', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1),
    'Multinomial NB': MultinomialNB()
}
results = []

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    metrics = {
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
    }
    results.append(metrics)

    print(classification_report(y_test, y_pred, target_names=['Real', 'Fake']))
    artifact_name = name.lower().replace(' ', '_')
    joblib.dump(model, MODEL_DIR / f"{artifact_name}.joblib")
    print('Saved model artifact for', name)

In [ ]:
results_df = pd.DataFrame(results).set_index('model')
results_df

## 6. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

## 7. Summary and next steps

- Classical models are trained and saved as joblib artifacts in `backend/models/`.
- Use these models to power the FastAPI inference endpoint in Week 4.
- Next step: implement BERT fine-tuning in `04_bert_fine_tuning.ipynb` and build the backend prediction API.